<a href="https://colab.research.google.com/github/zhangling297/deep-learning-with-python-notebooks/blob/master/Copy_of_Mat499_599_Assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 1

My outcome variable would be wine quality, and predicted varible include citricv acid ( increase freshness and flavor to wines), density( of water), and PH ( describes how acideic or basic a wine is on a scale from 0 being very acidic to 14 being very basic;

1.1 Use gradient descent with a learning rate of 0.01 to learn the parameters of each model


1.2 Produce a plot of Loss against epoch for each model, display th eplot with all there lines on the sam eset of axes

1.3 Discuss
(a) whether they all converged to the same minimum losso value ofr not,and hypothesize a relationship between loss and the number of parameters in a neural network

(b) Whether the number of epochs required to converge to its minimum differed between models, and whether there is an apparent relationship between this and number of parameters in a neural network?


In [ ]:
from ast import increment_lineno
from sklearn.base import ClassifierMixin
# import required packages
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import confusion_matrix, classification_report

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
%matplotlib inline

In [ ]:
# Load dataset
Wine = pd.read_csv('/_Winequality.csv')
Wine.head()

In [ ]:
Wine.info()

In [ ]:
# Invistigate data columns distribution
fig = plt.figure(figsize = (8,6))
sns.barplot(x = 'quality', y = 'citric acid', data = Wine)

**Queston 1 data exploration**
Above showing that citric acid does have a positive linear relationship with wine quality, as citric acide increase, wine quality assoicated numbers increases.

In [ ]:
# Check how density and ph associated with wine quality
fig = plt.figure(figsize = (8,6))
sns.barplot(x = 'quality', y = 'density', data = Wine)

**This** gives that density doesn't  give any specification to classsify the quality

In [ ]:
fig = plt.figure(figsize = (8,6))
sns.barplot(x = 'quality', y = 'pH', data = Wine)

**Same** as density, the PH won't give specification to classify the quality

In [ ]:
# If a classification task is needed later, define a separate binned quality column.
# For now, this cell will not modify the primary 'quality' column used for regression.
# bins = [2, 5, 8]
# group_names = ['low', 'high']
# Wine['quality_binned'] = pd.cut(Wine['quality'], bins = bins, labels = group_names)


In [ ]:
# Assign lables to quality variable
label_quality = LabelEncoder()

In [ ]:
# If a classification task is needed later, encode the binned quality column.
# label_quality = LabelEncoder()
# Wine['quality_binned'] = label_quality.fit_transform(Wine['quality_binned'])
# print(Wine['quality_binned'].value_counts())


In [ ]:
sns.countplot(Wine['quality'].value_counts())

In [ ]:
#Separate the dataset as response variable and feature variables
X = Wine.drop('quality', axis = 1)
y = Wine['quality'] # Use original continuous quality for regression

In [ ]:
#Train and test splitting of data
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [ ]:
#Applying Standard scaling to get optimized result
sc = StandardScaler()

In [ ]:
#Applying Standard scaling to get optimized result
#sc = StandardScaler()
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

**1.1 Linear Model**

In [ ]:
# 1. Linear Model
from sklearn.linear_model import SGDRegressor
import numpy as np

# Initialize lists to store loss history for plotting
loss_history_linear = []

# Define the Linear Model (SGDRegressor with linear loss)
# Set a learning rate of 0.01 and a fixed random_state for reproducibility
linear_model = SGDRegressor(loss='squared_error', learning_rate='constant', eta0=0.01, max_iter=1, warm_start=True, random_state=42)

# Train the model using gradient descent, tracking loss at each epoch
# We will iterate for a fixed number of epochs and record the loss
num_epochs = 1000 # You can adjust this value

for epoch in range(num_epochs):
    linear_model.fit(X_train, y_train)
    # Calculate loss (e.g., Mean Squared Error) on training data
    y_pred_train = linear_model.predict(X_train)
    loss = np.mean((y_train - y_pred_train)**2)
    loss_history_linear.append(loss)

print(f"Linear Model training complete. Final Loss: {loss_history_linear[-1]:.4f}")

**1.2 Single-hidden Layer NN With 12, 8, 4 neurons**

In [ ]:
# 2. Single-hidden layer Neural Network with 12 neurons
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Initialize list to store loss history for plotting
loss_history_nn_single = []

# Define the single-hidden layer Neural Network model
model_nn_single = keras.Sequential([
    layers.Dense(12, activation='relu', input_shape=(X_train.shape[1],)), # Hidden layer with 12 neurons and ReLU activation
    layers.Dense(1) # Output layer for regression (single neuron, no activation for linear output)
])

# Compile the model with Adam optimizer and Mean Squared Error loss
# We'll use a custom training loop to track loss per epoch with a fixed learning rate
# For simplicity, we'll re-implement a manual SGD-like update with learning_rate=0.01

# Manual training loop for tracking loss per epoch for neural network
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)
loss_fn = tf.keras.losses.MeanSquaredError()

num_epochs_nn = 1000 # Same number of epochs as the linear model for comparison

for epoch in range(num_epochs_nn):
    with tf.GradientTape() as tape:
        y_pred_train_nn = model_nn_single(X_train)
        loss_nn = loss_fn(y_train, y_pred_train_nn)

    gradients = tape.gradient(loss_nn, model_nn_single.trainable_weights)
    optimizer.apply_gradients(zip(gradients, model_nn_single.trainable_weights))
    loss_history_nn_single.append(loss_nn.numpy())

print(f"Single-hidden layer NN training complete. Final Loss: {loss_history_nn_single[-1]:.4f}")

**1.3 Three-hidden Layers NN wiht 12, 8, and 4 neurons**

In [ ]:
# 3. Three-Hidden layer Neural Network with 12, 8, and 4 neurons
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Initialize list to store loss history for plotting
loss_history_nn_three = []

# Define the three-hidden layer Neural Network model
model_nn_three = keras.Sequential([
    layers.Dense(12, activation='relu', input_shape=(X_train.shape[1],)), # First hidden layer
    layers.Dense(8, activation='relu'), # Second hidden layer
    layers.Dense(4, activation='relu'), # Third hidden layer
    layers.Dense(1) # Output layer for regression
])

# Manual training loop for tracking loss per epoch for neural network
optimizer_three = tf.keras.optimizers.SGD(learning_rate=0.01)
loss_fn_three = tf.keras.losses.MeanSquaredError()

num_epochs_nn_three = 1000 # Same number of epochs for consistency

for epoch in range(num_epochs_nn_three):
    with tf.GradientTape() as tape:
        y_pred_train_nn_three = model_nn_three(X_train)
        loss_nn_three = loss_fn_three(y_train, y_pred_train_nn_three)

    gradients_three = tape.gradient(loss_nn_three, model_nn_three.trainable_weights)
    optimizer_three.apply_gradients(zip(gradients_three, model_nn_three.trainable_weights))
    loss_history_nn_three.append(loss_nn_three.numpy())

print(f"Three-hidden layer NN training complete. Final Loss: {loss_history_nn_three[-1]:.4f}")

**Model Analysis - Loss against epoch for each model**
Epoch = 1000,

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(loss_history_linear, label='Linear Model (SGDRegressor)')
plt.plot(loss_history_nn_single, label='Single-hidden layer NN (12 neurons)')
plt.plot(loss_history_nn_three, label='Three-hidden layer NN (12, 8, 4 neurons)')
plt.title('Loss vs. Epoch for Different Models')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error Loss')
plt.legend()
plt.grid(True)
plt.show()

# Question 1 Discussion:
a) **The hypothesis is** there is a negetive association between loss and the nunber of parameters in a NN ** models don't share the same loss, with the three layer Neural nertwork obtained the lowest loss at 0.3965 and single layer about o.4077; we can assume increased paramerters in a NN can decrease the loss value when epoches reach to certain numbers, then remain the same or very slightly change as epoches number increases.

b) **** **As number of parameter increase and Epoches increases, loss diseases first and then becomes steady  ** number of epochs less than 200, all modeles appears signifiant loss reduction; from 200 to 300 epochs, all loss curves for all models are becoming flat, meaning there will be minimum loss changes after 300 epochs; More numbers of parameters in a neural network will require longer runtime to converge than simple linear models.

# Question 2  

Revisit the 3-hidden layer model , change the number of neurons to 16, 8, and 4, for the hidden layers in that order ( may define a new class, and learn the model with 5 different learning rates: 0.00001, 0.0001, 0.001, 0.01, 0.1)  Task 1) is create a plot of Loss Against Epoch and plot  on the sam eaxis. - Task2) Create a plot of Loss against Epoch and plot each of the 5 variants on the same axis; Task 3) discuss the impact leanring rate has on a) the minimum loss and b) the number of epochs required for the model to converge.`

**3-hidden layer model with 16, 8, and 4 neurons and lr = 0.00001, 0.0001, 0.00, 0.01, and 0.1**

In [ ]:
# Revisit the 3-hidden layer model with 16, 8, and 4 neurons, and learn with 5 different learning rates
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

learning_rates = [0.00001, 0.0001, 0.001, 0.01, 0.1]
all_loss_histories = {}
num_epochs_lr = 1000 # Same number of epochs for consistency

for lr in learning_rates:
    print(f"\nTraining with learning rate: {lr}")
    # Define the three-hidden layer Neural Network model for each learning rate
    model_nn_lr = keras.Sequential([
        layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)), # First hidden layer
        layers.Dense(8, activation='relu'), # Second hidden layer
        layers.Dense(4, activation='relu'), # Third hidden layer
        layers.Dense(1) # Output layer for regression
    ])

    # Manual training loop for tracking loss per epoch
    optimizer_lr = tf.keras.optimizers.SGD(learning_rate=lr)
    loss_fn_lr = tf.keras.losses.MeanSquaredError()
    current_loss_history = []

    for epoch in range(num_epochs_lr):
        with tf.GradientTape() as tape:
            y_pred_train_nn_lr = model_nn_lr(X_train)
            loss_nn_lr = loss_fn_lr(y_train, y_pred_train_nn_lr)

        gradients_lr = tape.gradient(loss_nn_lr, model_nn_lr.trainable_weights)
        optimizer_lr.apply_gradients(zip(gradients_lr, model_nn_lr.trainable_weights))
        current_loss_history.append(loss_nn_lr.numpy())

    all_loss_histories[str(lr)] = current_loss_history
    print(f"Training complete for LR {lr}. Final Loss: {current_loss_history[-1]:.4f}")

# Task 1) is create a plot of Loss Against Epoch and plot each of the 5 variants on the same axis
plt.figure(figsize=(12, 8))
for lr_str, history in all_loss_histories.items():
    plt.plot(history, label=f'LR: {lr_str}')

plt.title('Loss vs. Epoch for Three-Hidden Layer NN with Different Learning Rates')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error Loss')
plt.legend()
plt.grid(True)
plt.show()

# Problem 2 Discussion:

**(a) ** Higher learning rates 0.1 lead to steepest convergence and requires fewer epochs to reach a local minimum loss; this can potentially cause training process unstable and might miss local minimals; while with lr valued 0.00001, its progress in reducing loss is very small and takes much longer run time to converge to a competitive minimum.
**(b)** Smaller learning rate requires large amount of epochs to potentially reach a lower loss while larger learning rate needs much more epochs for the model to converge

#Question 3 :

**3.1 Split the data in half at random** and relearn the three models from Question 1 "Use linear model, a single-hidden layer neural network with 12 neurons, and a three-hidden layer neural network with 12, 8, and 4 neurons in the hidden layers to preidct wine quality & Use gradient descent with a learning rate of 0.01 to learn the parameters of each model

**3.2 Split plot of Loss against epoch for each model**, display th eplot with all there lines on the sam eset of axes


**3.3 Discuss** (a) whether they all converged to the same minimum losso value ofr not,and hypothesize a relationship between loss and the number of parameters in a neural network
(b) Whether the number of epochs required to converge to its minimum differed between models, and whether there is an apparent relationship between this and number of parameters in a neural network?. Addition, discuss I: Does the quantity of data change the value of the loss that the models converge on? II: Does it affect the models similarly or differently? III: Draw a hypothesis from these results.

In [ ]:
 #Single-hidden layer Neural Network with 12 neurons (Half Data)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Initialize list to store loss history for plotting
loss_history_nn_single_half= []

# Define the single-hidden layer Neural Network model
model_nn_single_half = keras.Sequential([
    layers.Dense(12, activation='relu', input_shape=(X_train.shape[1],)), # Hidden layer with 12 neurons and ReLU activation
    layers.Dense(1) # Output layer for regression (single neuron, no activation for linear output)
])

# Manual training loop for tracking loss per epoch for neural network
optimizer_single_half= tf.keras.optimizers.SGD(learning_rate=0.01)
loss_fn_single_half = tf.keras.losses.MeanSquaredError()

num_epochs_nn_single= 1000 # Same number of epochs for consistency

for epoch in range(num_epochs_nn_single):
    with tf.GradientTape() as tape:
        y_pred_train_nn_single= model_nn_single_half(X_train)
        loss_nn_single_half= loss_fn_single_half(y_train, y_pred_train_nn_single) # Corrected variable name here

    gradients_single_half= tape.gradient(loss_nn_single, model_nn_single.trainable_weights)
    optimizer_single.apply_gradients(zip(gradients_single, model_nn_single.trainable_weights))
    loss_history_nn_single_half.append(loss_nn_single.numpy())

print(f"Single-hidden layer NN training complete (Wine Data). Final Loss: {loss_history_nn_single[-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(loss_history_linear, label='Linear Model (Half Data)')
plt.plot(loss_history_nn_single_half, label='Single-hidden layer NN (Half Data)')
plt.plot(loss_history_nn_three_half, label='Three-hidden layer NN (Half Data)')
plt.title('Loss vs. Epoch for Models with Half Data')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error Loss')
plt.legend()
plt.grid(True)
plt.show()

# Question 3 Discussion

(1). Yes - The quality of data change the value of the loss that the movdels converge on. The half data signicate decreases the number of Epoch needed for the model converge on minumal loss. High quality data greatly reduce runtime and high efficient to train and test model.

(2). High quality of data with good algerithems can reach to loss minumum faster than low quality data, and only need small number of epochs for the model to converge.